# Real-time edge measurements

This notebook exercises the local latency and cost fixture. It does not download a model or claim a device benchmark.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/15-real-time-edge')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l15_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
if not module.TORCH_AVAILABLE:
    samples = sorted([1.1, 1.4, 1.8, 2.0])
    p95 = samples[min(len(samples) - 1, int(0.95 * len(samples)))]
    assert p95 == 2.0
    print({'Build-It': 'Rust measurement contract', 'p95_fixture_ms': p95, 'Use-It': 'PyTorch skipped cleanly'})
else:
    model = module.TinyDenseBackbone(num_classes=4).eval()
    report = module.measure_latency(model, (1, 3, 32, 32), warmup=1, iters=2)
    assert report['p50_ms'] >= 0 and report['p95_ms'] >= report['p50_ms']
    print(report)

Use the report as a local engineering artifact: compare `params`, estimated FLOPs, and tail latency before making a deployment decision.